# NLRexpress on Google Colab

This notebook runs [NLRexpress](https://github.com/eliza-m/NLRexpress) on Google Colab for single-sequence or multi-sequence input. You can provide protein sequences directly, or provide CDS sequences that will be translated before running NLRexpress.

The original NLRexpress outputs are left unchanged. MHD D-to-V mutation candidates are written to separate CSV files.

If CDS sequences are provided, the notebook can also write a CodonDomesticate-ready CSV containing the original CDS plus the inferred `aa_change` value, so the next notebook can perform domestication and MHD D-to-V mutation in one batch run.


## Suggested Workflow

### If you have CDS sequences

1. Run NLRexpress here with `input_sequence_type = "CDS"`.
2. Download the original NLRexpress result archive.
3. Download the separate MHD D-to-V mutation candidate table.
4. Download the CodonDomesticate-ready CSV generated from your CDS input.
5. Open the CodonDomesticate notebook and upload that CSV for batch domestication/mutation.

### If you only have protein sequences

1. Run NLRexpress here with `input_sequence_type = "protein"`.
2. Download the original NLRexpress result archive and MHD D-to-V candidate table.
3. CDS domestication cannot be performed until matching CDS sequences are available.

[![Open CodonDomesticate In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YuSugihara/CodonDomesticate/blob/main/notebooks/Domesticate_CDS_Colab.ipynb)


In [ ]:
#@title Install isolated NLRexpress environment
%%capture
!pip install -q pandas biopython
!if [ ! -x /content/bin/micromamba ]; then cd /content && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba; fi
!if [ ! -d /content/nlrexpress_env ]; then /content/bin/micromamba create -y -p /content/nlrexpress_env -c conda-forge python=3.9 click numpy=1.22 pandas=1.4 scipy=1.8 scikit-learn=0.24.2 joblib biopython hmmer; fi


In [ ]:
#@title Download NLRexpress and predictor models
from pathlib import Path
import os
import subprocess

NLR_REPO = Path("NLRexpress")
MODELS_DIR = NLR_REPO / "models"

if not NLR_REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/eliza-m/NLRexpress.git", str(NLR_REPO)], check=True)
else:
    print("NLRexpress repository already exists.")

if not MODELS_DIR.exists() or not list(MODELS_DIR.glob("*.pkl")):
    subprocess.run(["wget", "-q", "https://nlrexpress.biochim.ro/datasets/models.tar.gz", "-O", str(NLR_REPO / "models.tar.gz")], check=True)
    subprocess.run(["tar", "-xf", str(NLR_REPO / "models.tar.gz"), "-C", str(NLR_REPO)], check=True)
else:
    print("NLRexpress predictor models already exist.")

print("NLRexpress is ready:", NLR_REPO.resolve())


In [ ]:
#@title Helper functions
from pathlib import Path
import csv
import os
import re
import shutil
import subprocess
import textwrap
import zipfile

import pandas as pd
from google.colab import files

VALID_AA = set("ACDEFGHIKLMNPQRSTVWYBXZJUO*-")
DNA_BASES = set("ACGTUacgtu")
STANDARD_GENETIC_CODE = {
    "TTT": "F", "TTC": "F", "TTA": "L", "TTG": "L",
    "TCT": "S", "TCC": "S", "TCA": "S", "TCG": "S",
    "TAT": "Y", "TAC": "Y", "TAA": "*", "TAG": "*",
    "TGT": "C", "TGC": "C", "TGA": "*", "TGG": "W",
    "CTT": "L", "CTC": "L", "CTA": "L", "CTG": "L",
    "CCT": "P", "CCC": "P", "CCA": "P", "CCG": "P",
    "CAT": "H", "CAC": "H", "CAA": "Q", "CAG": "Q",
    "CGT": "R", "CGC": "R", "CGA": "R", "CGG": "R",
    "ATT": "I", "ATC": "I", "ATA": "I", "ATG": "M",
    "ACT": "T", "ACC": "T", "ACA": "T", "ACG": "T",
    "AAT": "N", "AAC": "N", "AAA": "K", "AAG": "K",
    "AGT": "S", "AGC": "S", "AGA": "R", "AGG": "R",
    "GTT": "V", "GTC": "V", "GTA": "V", "GTG": "V",
    "GCT": "A", "GCC": "A", "GCA": "A", "GCG": "A",
    "GAT": "D", "GAC": "D", "GAA": "E", "GAG": "E",
    "GGT": "G", "GGC": "G", "GGA": "G", "GGG": "G",
}
MICROMAMBA = Path("/content/bin/micromamba")
NLR_ENV_PREFIX = Path("/content/nlrexpress_env")


def clean_protein_sequence(seq):
    seq = "".join(str(seq).split()).upper()
    invalid = sorted(set(seq) - VALID_AA)
    if invalid:
        raise ValueError(f"Protein sequence contains invalid characters: {''.join(invalid)}")
    return seq.replace("-", "")


def clean_cds_sequence(seq):
    cds = "".join(str(seq).split()).upper().replace("U", "T")
    invalid = sorted(set(cds) - set("ACGT"))
    if invalid:
        raise ValueError(f"CDS contains invalid DNA bases: {''.join(invalid)}")
    if len(cds) % 3 != 0:
        raise ValueError("CDS length is not a multiple of 3.")
    return cds


def translate_cds(cds, trim_terminal_stop=True):
    cds = clean_cds_sequence(cds)
    protein = "".join(STANDARD_GENETIC_CODE[cds[i:i + 3]] for i in range(0, len(cds), 3))
    internal = protein[:-1] if protein.endswith("*") else protein
    if "*" in internal:
        raise ValueError("CDS contains an internal stop codon.")
    if trim_terminal_stop and protein.endswith("*"):
        protein = protein[:-1]
    return protein


def write_fasta(records, fasta_path):
    fasta_path = Path(fasta_path)
    with fasta_path.open("w") as handle:
        for name, seq in records:
            safe_name = re.sub(r"\s+", "_", str(name).strip()) or "sequence"
            clean_seq = clean_protein_sequence(seq)
            handle.write(f">{safe_name}\n")
            for i in range(0, len(clean_seq), 80):
                handle.write(clean_seq[i:i + 80] + "\n")
    return fasta_path


def run_nlrexpress(input_fasta, outdir, module="all", outformat="all", cpunum=2):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    cmd = [
        str(MICROMAMBA), "run", "-p", str(NLR_ENV_PREFIX),
        "python", "nlrexpress.py",
        "--input", str(Path(input_fasta).resolve()),
        "--outdir", str(outdir.resolve()),
        "--module", module,
        "--outformat", outformat,
        "--cpunum", str(cpunum),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd="NLRexpress", check=True)
    return outdir


def zip_directory(directory, zip_path):
    directory = Path(directory)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for path in directory.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(directory.parent))
    return zip_path


def parse_mhd_hits(short_output_path):
    hits = []
    short_output_path = Path(short_output_path)
    if not short_output_path.exists():
        return hits
    with short_output_path.open() as handle:
        for line in handle:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 8:
                continue
            if parts[2] != "MHD":
                continue
            protein_name = parts[0]
            motif_start = int(parts[1])
            probability = float(parts[3])
            motif_seq = parts[7]
            d_offset = motif_seq.rfind("D")
            if d_offset < 0:
                d_offset = len(motif_seq) - 1
            d_position = motif_start + d_offset
            hits.append({
                "name": protein_name,
                "motif_start": motif_start,
                "motif": "MHD",
                "probability": probability,
                "motif_seq": motif_seq,
                "d_position": d_position,
                "aa_change": f"D{d_position}V",
                "source_file": str(short_output_path),
            })
    return hits


def collect_mhd_hits(outdir):
    all_hits = []
    for path in Path(outdir).glob("*.short.output.txt"):
        all_hits.extend(parse_mhd_hits(path))
    return pd.DataFrame(all_hits)


def download_results(outdir, prefix):
    zip_path = zip_directory(outdir, f"{prefix}_nlrexpress_results.zip")
    files.download(str(zip_path))
    return zip_path


def records_from_csv(input_path, sequence_type):
    df = pd.read_csv(input_path)
    if not {"name", "sequence"}.issubset(df.columns):
        raise ValueError("CSV must contain 'name' and 'sequence' columns.")
    protein_records = []
    cds_records = []
    for _, row in df.iterrows():
        name = str(row["name"]).strip()
        sequence = str(row["sequence"])
        if sequence_type == "CDS":
            cds = clean_cds_sequence(sequence)
            protein_records.append((name, translate_cds(cds)))
            cds_records.append({"name": name, "sequence": cds})
        else:
            protein_records.append((name, clean_protein_sequence(sequence)))
    cds_df = pd.DataFrame(cds_records) if cds_records else None
    return protein_records, cds_df


def parse_fasta_records(input_path):
    records = []
    name = None
    chunks = []
    with open(input_path) as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    records.append((name, "".join(chunks)))
                name = line[1:].split()[0]
                chunks = []
            else:
                chunks.append(line)
    if name is not None:
        records.append((name, "".join(chunks)))
    if not records:
        raise ValueError("No FASTA records found.")
    return records


def records_from_fasta(input_path, sequence_type):
    raw_records = parse_fasta_records(input_path)
    protein_records = []
    cds_records = []
    for name, sequence in raw_records:
        if sequence_type == "CDS":
            cds = clean_cds_sequence(sequence)
            protein_records.append((name, translate_cds(cds)))
            cds_records.append({"name": name, "sequence": cds})
        else:
            protein_records.append((name, clean_protein_sequence(sequence)))
    cds_df = pd.DataFrame(cds_records) if cds_records else None
    return protein_records, cds_df


def write_codon_domesticate_handoff(cds_df, candidate_df, output_csv):
    if cds_df is None or cds_df.empty:
        print("No CDS input was provided, so CodonDomesticate handoff CSV was not created.")
        return None
    if candidate_df is None or candidate_df.empty:
        print("WARNING: No MHD motif was found. The CodonDomesticate handoff CSV will be written with an empty aa_change column.")
        merged = cds_df.copy()
        merged["aa_change"] = ""
    else:
        first_candidates = candidate_df.sort_values(["name", "probability"], ascending=[True, False]).drop_duplicates("name")
        first_candidates = first_candidates.rename(columns={"aa_change": "mhd_dv_aa_change"})
        merged = cds_df.merge(
            first_candidates[["name", "mhd_dv_aa_change", "motif_start", "motif_seq", "d_position", "probability"]],
            on="name",
            how="left",
        )
        missing_names = merged.loc[merged["mhd_dv_aa_change"].isna(), "name"].astype(str).tolist()
        if missing_names:
            preview = ", ".join(missing_names[:10])
            suffix = "" if len(missing_names) <= 10 else f" ... and {len(missing_names) - 10} more"
            print(f"WARNING: No MHD motif was found for {len(missing_names)} CDS record(s): {preview}{suffix}. Their aa_change values will be empty.")
        merged["aa_change"] = merged["mhd_dv_aa_change"].fillna("")
        merged = merged.drop(columns=["mhd_dv_aa_change"])
    merged.to_csv(output_csv, index=False)
    files.download(output_csv)
    print("Ready for CodonDomesticate batch input:", output_csv)
    return merged


## Single Sequence Run

Paste one protein or CDS sequence and run NLRexpress. Use `module = "nbs"` if you only need NB-ARC/NBS motifs including MHD.


In [ ]:
#@title Configure single sequence input
input_sequence_type = "CDS" #@param ["CDS", "protein"]
sequence_name = "example_NLR" #@param {type:"string"}
sequence = "" #@param {type:"string"}
module = "nbs" #@param ["nbs", "all", "cc", "tir", "lrr"]
outformat = "all" #@param ["all", "short", "long"]
cpunum = 2 #@param {type:"integer"}


In [ ]:
#@title Run NLRexpress for one sequence
if not sequence.strip():
    raise ValueError("Please paste a protein or CDS sequence into sequence.")

single_cds_df = None
if input_sequence_type == "CDS":
    single_cds = clean_cds_sequence(sequence)
    single_protein = translate_cds(single_cds)
    single_cds_df = pd.DataFrame([{"name": sequence_name, "sequence": single_cds}])
else:
    single_protein = clean_protein_sequence(sequence)

single_fasta = write_fasta([(sequence_name, single_protein)], "single_input.fa")
single_outdir = run_nlrexpress(single_fasta, "nlrexpress_single_output", module=module, outformat=outformat, cpunum=cpunum)

single_mhd_df = collect_mhd_hits(single_outdir)
print("MHD D-to-V candidates:")
display(single_mhd_df if not single_mhd_df.empty else pd.DataFrame(columns=["name", "motif_start", "motif_seq", "d_position", "aa_change", "probability"]))

single_mhd_path = "single_mhd_dv_candidates.csv"
single_mhd_df.to_csv(single_mhd_path, index=False)
files.download(single_mhd_path)

if single_cds_df is not None:
    single_handoff_df = write_codon_domesticate_handoff(single_cds_df, single_mhd_df, "single_codon_domesticate_input_with_mhd_dv.csv")
    display(single_handoff_df)

download_results(single_outdir, "single")


## Multiple Sequence Run

Upload a CSV with `name` and `sequence` columns, or upload a FASTA file containing one or more sequences. Set `multi_input_sequence_type` to `CDS` if the uploaded sequences are CDS; the notebook will translate them before running NLRexpress and keep the CDS for CodonDomesticate handoff.


In [ ]:
#@title Download multi-sequence CSV templates
protein_template_path = "nlrexpress_protein_template.csv"
with open(protein_template_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "sequence"])
    writer.writeheader()
    writer.writerow({"name": "example_NLR_1", "sequence": "MA..."})
files.download(protein_template_path)

cds_template_path = "nlrexpress_cds_template.csv"
with open(cds_template_path, "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["name", "sequence"])
    writer.writeheader()
    writer.writerow({"name": "example_NLR_1", "sequence": "ATG..."})
files.download(cds_template_path)

print("Templates written:", protein_template_path, cds_template_path)


In [ ]:
#@title Upload sequence CSV or FASTA
uploaded = files.upload()
input_path = next(iter(uploaded))
print("Uploaded:", input_path)


In [ ]:
#@title Configure multiple sequence run
multi_input_sequence_type = "CDS" #@param ["CDS", "protein"]
multi_module = "nbs" #@param ["nbs", "all", "cc", "tir", "lrr"]
multi_outformat = "all" #@param ["all", "short", "long"]
multi_cpunum = 2 #@param {type:"integer"}


In [ ]:
#@title Run NLRexpress for multiple sequences
input_path = Path(input_path)
if input_path.suffix.lower() == ".csv":
    protein_records, multi_cds_df = records_from_csv(input_path, multi_input_sequence_type)
else:
    protein_records, multi_cds_df = records_from_fasta(input_path, multi_input_sequence_type)

multi_fasta = write_fasta(protein_records, "multi_input.fa")
multi_outdir = run_nlrexpress(multi_fasta, "nlrexpress_multi_output", module=multi_module, outformat=multi_outformat, cpunum=multi_cpunum)

multi_mhd_df = collect_mhd_hits(multi_outdir)
print("MHD D-to-V candidates:")
display(multi_mhd_df if not multi_mhd_df.empty else pd.DataFrame(columns=["name", "motif_start", "motif_seq", "d_position", "aa_change", "probability"]))

multi_mhd_path = "multi_mhd_dv_candidates.csv"
multi_mhd_df.to_csv(multi_mhd_path, index=False)
files.download(multi_mhd_path)

if multi_cds_df is not None:
    multi_handoff_df = write_codon_domesticate_handoff(multi_cds_df, multi_mhd_df, "multi_codon_domesticate_input_with_mhd_dv.csv")
    display(multi_handoff_df)

download_results(multi_outdir, "multi")


## Prepare CodonDomesticate Input with MHD D-to-V Mutations

If you ran NLRexpress with CDS input, the notebook already creates a CodonDomesticate-ready CSV. Use this optional section only when you ran NLRexpress with protein input and want to upload a separate matching CDS CSV afterward. The `name` values should match the protein names in the NLRexpress MHD table. The notebook writes a new CSV with an `aa_change` column such as `D489V`; it does not modify the original NLRexpress output files.


In [ ]:
#@title Optional: upload CDS CSV and merge MHD D-to-V candidates
uploaded_cds = files.upload()
cds_csv_path = next(iter(uploaded_cds))
cds_df = pd.read_csv(cds_csv_path)
if not {"name", "sequence"}.issubset(cds_df.columns):
    raise ValueError("CDS CSV must contain 'name' and 'sequence' columns.")
cds_df = pd.DataFrame([{"name": row["name"], "sequence": clean_cds_sequence(row["sequence"])} for _, row in cds_df.iterrows()])

candidate_df = None
if "multi_mhd_df" in globals() and not multi_mhd_df.empty:
    candidate_df = multi_mhd_df
elif "single_mhd_df" in globals() and not single_mhd_df.empty:
    candidate_df = single_mhd_df
else:
    raise ValueError("No MHD D-to-V candidates are available. Run NLRexpress first.")

manual_handoff_df = write_codon_domesticate_handoff(cds_df, candidate_df, "codon_domesticate_input_with_mhd_dv.csv")
display(manual_handoff_df)
